# Pipeline 3 — Explainable Boosting Machine (Maturo et al., 2025)

Methodology: Explainable Boosting Machine (InterpretML) following Maturo et al. (2025), with:

- Sampling-strategy tournament: sector-stratified undersampling (`1:1`, `1:3`, `1:5`) versus class/sample weighting. The winner is selected by **validation PR-AUC**, which is more informative than accuracy under the approximately 0.18 % bankruptcy base rate.
- Out-of-time split — train 2013–2016, validation 2017, test 2018–2019.
- Optuna TPE hyperparameter search on the winning sampling strategy.
- Lag-1 / lag-2 / diff-1 / diff-2 features from notebook 00, giving the row-wise EBM temporal context without requiring a recurrent architecture.
- Manual sigmoid/Platt-style calibration on the validation set for calibrated probabilities.
- Threshold selected by Youden-J on the validation predictions.
- Global interpretability via EBM term importances, shape functions for the top main effects and inspection of the strongest pairwise interactions.
- Extended diagnostics: bootstrap confidence intervals, feature-stability across seeds, per-year decomposition and calibration metrics.

Inputs: `data_ebm.csv`. Outputs: `results_ebm.pkl` plus an extended `ebm_v7_results.pkl` payload for downstream model comparison.


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss, classification_report
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.class_weight import compute_sample_weight

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    print("InterpretML OK")
except ImportError:
    raise ImportError("Nainštalujte: pip install interpret")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
    print("Optuna OK")
except ImportError:
    OPTUNA_OK = False
    print("Optuna nie je nainštalovaná — tuning bude preskočený")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
DATA_DIR = os.getcwd()
N_TPE_TRIALS = 30


## 1. Load the EBM-ready dataset

Reads `data_ebm.csv` (raw items + ratios + lag/diff features + non-financials + industry dummies + macro), sorted by `(id, rok)`. Reports shape, target rate and confirms that lag/diff columns are present.

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, 'data_ebm.csv'))
data = data.sort_values(['id', 'rok']).reset_index(drop=True)

print(f"EBM dataset: {data.shape}")
print(f"Target rate: {data['target'].mean()*100:.3f}%")
print(f"Lag features: {sum(1 for c in data.columns if '_lag' in c)}")
print(f"Diff features: {sum(1 for c in data.columns if '_diff' in c)}")
print(f"Roky: {sorted(data['rok'].unique())}")

# Feature stĺpce (všetko okrem id, rok, target)
exclude = ['id', 'rok', 'target']
feature_cols = [c for c in data.columns if c not in exclude]
print(f"\nCelkovo features: {len(feature_cols)}")


## 2. Out-of-time split

Splits the panel into train 2013–2016, validation 2017, test 2018–2019. Stores `feature_cols`, `X_*`, `y_*` and the corresponding `id` / `rok` / `sector` arrays for sampling and per-year analysis later.

In [ ]:
train = data[data['rok'].between(2013, 2016)].copy()
valid = data[data['rok'] == 2017].copy()
test  = data[data['rok'].between(2018, 2019)].copy()

X_train_full = train[feature_cols]
y_train_full = train['target']
X_valid      = valid[feature_cols]
y_valid      = valid['target']
X_test       = test[feature_cols]
y_test       = test['target']

# Odvetvie pre stratifikovaný undersampling.
# Nepoužívame prvé znaky ID, ale odvetvové dummy premenné vytvorené v pipeline 00.
industry_cols = [c for c in feature_cols if c.startswith('industry_') or c.startswith('nace_')]

if industry_cols:
    sector_train = train[industry_cols].idxmax(axis=1).fillna('industry_missing')
else:
    sector_train = pd.Series('industry_missing', index=train.index)

print(f"TRAIN: {len(train):>8,} | failures: {y_train_full.sum():>5,} | rate: {y_train_full.mean()*100:.3f}%")
print(f"VALID: {len(valid):>8,} | failures: {y_valid.sum():>5,} | rate: {y_valid.mean()*100:.3f}%")
print(f"TEST:  {len(test):>8,} | failures: {y_test.sum():>5,}  | rate: {y_test.mean()*100:.3f}%")


## 3. Sampling strategies

Four strategies are evaluated against each other on validation PR-AUC:

- `undersample_1_1` — random under-sampling at 1:1 (failures : healthy).
- `undersample_1_3` — random under-sampling at 1:3.
- `undersample_1_5` — random under-sampling at 1:5.
- `weighted` — no resampling; the EBM sample weights are set to
  `n_neg / n_pos`.

Under-sampling is sector-stratified to preserve the industry mix.

In [ ]:
def stratified_undersample(X, y, sector, majority_ratio=1, random_state=42):
    """Stratifikovaný undersampling majoritynej triedy podľa sektora."""
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y)).reset_index(drop=True)
    sector = pd.Series(np.asarray(sector)).reset_index(drop=True)

    min_idx = y[y == 1].index
    maj_idx = y[y == 0].index
    n_min   = len(min_idx)
    n_keep  = min(n_min * majority_ratio, len(maj_idx))

    rng = np.random.RandomState(random_state)
    maj_df = pd.DataFrame({'idx': maj_idx, 'sec': sector[maj_idx].values})
    sec_frac = maj_df['sec'].value_counts(normalize=True)

    kept = []
    for sec, frac in sec_frac.items():
        sec_idx = maj_df[maj_df['sec'] == sec]['idx'].values
        n_sec   = max(1, int(round(frac * n_keep)))
        n_sec   = min(n_sec, len(sec_idx))
        kept.extend(rng.choice(sec_idx, size=n_sec, replace=False).tolist())
    kept = list(set(kept))

    final_idx = np.array(sorted(list(min_idx) + kept))
    return X.iloc[final_idx].reset_index(drop=True), y.iloc[final_idx].reset_index(drop=True)

def make_sample_weights(y):
    """Inversely proportional class weights."""
    y = np.asarray(y)
    n_pos = y.sum(); n_neg = len(y) - n_pos
    w_pos = len(y) / (2 * n_pos) if n_pos > 0 else 1.0
    w_neg = len(y) / (2 * n_neg) if n_neg > 0 else 1.0
    return np.where(y == 1, w_pos, w_neg)

# Baseline EBM parametre pre férové porovnanie stratégií
BASELINE_PARAMS = {
    'interactions':   10,
    'learning_rate':  0.01,
    'max_bins':       256,
    'max_leaves':     3,
    'min_samples_leaf': 5,
    'random_state':   RANDOM_SEED,
    'n_jobs':        -1,
}

strategies = {
    'undersample_1_1': dict(ratio=1),
    'undersample_1_3': dict(ratio=3),
    'undersample_1_5': dict(ratio=5),
    'weighted':        dict(ratio=None),
}

results_strategies = []
models_strat = {}
probs_strat  = {}

for strat_name, cfg in strategies.items():
    print(f"\n--- Stratégia: {strat_name} ---")
    if cfg['ratio'] is not None:
        X_s, y_s = stratified_undersample(
            X_train_full, y_train_full, sector_train,
            majority_ratio=cfg['ratio'], random_state=RANDOM_SEED
        )
        sw = None
    else:
        X_s, y_s = X_train_full.copy(), y_train_full.copy()
        sw = make_sample_weights(y_s)

    ebm = ExplainableBoostingClassifier(**BASELINE_PARAMS)
    if sw is not None:
        ebm.fit(X_s, y_s, sample_weight=sw)
    else:
        ebm.fit(X_s, y_s)

    y_prob_v = ebm.predict_proba(X_valid)[:, 1]
    y_prob_t = ebm.predict_proba(X_test)[:, 1]

    roc_v = roc_auc_score(y_valid, y_prob_v)
    pr_v  = average_precision_score(y_valid, y_prob_v)
    roc_t = roc_auc_score(y_test, y_prob_t)
    pr_t  = average_precision_score(y_test, y_prob_t)

    print(f"  Valid  ROC-AUC={roc_v:.4f}  PR-AUC={pr_v:.4f}")
    print(f"  Test   ROC-AUC={roc_t:.4f}  PR-AUC={pr_t:.4f}")
    print(f"  Train size: {len(X_s):,} (positives: {int(y_s.sum())})")

    results_strategies.append({
        'strategy': strat_name,
        'valid_roc': roc_v, 'valid_pr': pr_v,
        'test_roc':  roc_t, 'test_pr':  pr_t,
    })
    models_strat[strat_name] = ebm
    probs_strat[strat_name]  = y_prob_t

strat_df = pd.DataFrame(results_strategies).sort_values('valid_pr', ascending=False)
print("\n=== POROVNANIE STRATÉGIÍ (zoradené podľa Valid PR-AUC) ===")
print(strat_df.to_string(index=False))

WINNER = strat_df.iloc[0]['strategy']
print(f"\n→ Víťazná stratégia: {WINNER}")


## 4. Optuna TPE hyperparameter tuning

Runs Bayesian optimisation (Tree-structured Parzen Estimator) on the winning sampling strategy, with **validation PR-AUC** as the objective. Optimised hyperparameters: `max_bins`, `learning_rate`, `interactions`, `max_leaves` and `min_samples_leaf`. Reports the best PR-AUC and the winning configuration.

In [ ]:
# Pripraviť dáta pre víťazovu stratégiu
cfg_winner = strategies[WINNER]
if cfg_winner['ratio'] is not None:
    X_tune, y_tune = stratified_undersample(
        X_train_full, y_train_full, sector_train,
        majority_ratio=cfg_winner['ratio'], random_state=RANDOM_SEED
    )
    sw_tune = None
else:
    X_tune, y_tune = X_train_full.copy(), y_train_full.copy()
    sw_tune = make_sample_weights(y_tune)

if OPTUNA_OK:
    def tpe_objective(trial):
        params = {
            'interactions':     trial.suggest_int('interactions', 5, 25),
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'max_bins':         trial.suggest_int('max_bins', 128, 512),
            'max_leaves':       trial.suggest_int('max_leaves', 2, 5),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 30),
            'random_state':     RANDOM_SEED,
            'n_jobs':          -1,
        }
        ebm = ExplainableBoostingClassifier(**params)
        if sw_tune is not None:
            ebm.fit(X_tune, y_tune, sample_weight=np.asarray(sw_tune))
        else:
            ebm.fit(X_tune, y_tune)
        prob_v = ebm.predict_proba(X_valid)[:, 1]
        return average_precision_score(y_valid, prob_v)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(tpe_objective, n_trials=N_TPE_TRIALS, show_progress_bar=True)
    best_params = dict(study.best_params)
    best_params.update({'random_state': RANDOM_SEED, 'n_jobs': -1})
    print(f"\nBest PR-AUC (valid): {study.best_value:.4f}")
    print(f"Best params: {best_params}")
else:
    best_params = BASELINE_PARAMS
    print("Optuna nedostupná → používam baseline parametre")


## 5. Final EBM + Platt scaling calibration

Fits the final EBM on a `pandas.DataFrame` (preserving feature names — required for the explainability output to be human-readable). Calibration is fitted manually on the validation predictions: a logistic regression uses the raw EBM validation probabilities as a single predictor and maps them onto calibrated probabilities for validation and test.

In [ ]:
# Finálny EBM — fitujeme NAD DATAFRAME (nie numpy!) pre zachovanie názvov features
# KRITICKÉ pre správne fungovanie EBM explainability výstupov
# Manuálny Platt scaling (sklearn-version-safe)
from sklearn.linear_model import LogisticRegression as _LRCalib

cfg_winner = strategies[WINNER]

if cfg_winner['ratio'] is not None:
    X_final, y_final = stratified_undersample(
        X_train_full, y_train_full, sector_train,
        majority_ratio=cfg_winner['ratio'],
        random_state=RANDOM_SEED
    )
    sw_final = None
else:
    X_final, y_final = X_train_full.copy(), y_train_full.copy()
    sw_final = make_sample_weights(y_final)

ebm_final = ExplainableBoostingClassifier(**best_params)

if sw_final is not None:
    ebm_final.fit(X_final, y_final, sample_weight=sw_final)
else:
    ebm_final.fit(X_final, y_final)

y_prob_valid_raw = ebm_final.predict_proba(X_valid)[:, 1]
y_prob_test_raw  = ebm_final.predict_proba(X_test)[:, 1]

_platt = _LRCalib(C=1e10, solver='lbfgs', max_iter=1000)
_platt.fit(y_prob_valid_raw.reshape(-1, 1), y_valid)

y_prob_valid_cal = _platt.predict_proba(y_prob_valid_raw.reshape(-1, 1))[:, 1]
y_prob_test_cal  = _platt.predict_proba(y_prob_test_raw.reshape(-1, 1))[:, 1]

print(f"Po Platt scaling:")
print(f"  Test ROC-AUC: {roc_auc_score(y_test, y_prob_test_cal):.4f}")
print(f"  Test PR-AUC:  {average_precision_score(y_test, y_prob_test_cal):.4f}")
print(f"  Brier score:  {brier_score_loss(y_test, y_prob_test_cal):.4f}")

y_prob_test  = y_prob_test_cal
y_prob_valid = y_prob_valid_cal


## 6. Youden-J threshold + evaluation

Picks the threshold that maximises `TPR − FPR` on validation, then reports test-set ROC-AUC, PR-AUC, KS, Brier, recall and precision at that threshold.

In [ ]:
fpr_v, tpr_v, thr_v = roc_curve(y_valid, y_prob_valid)
youden_idx       = np.argmax(tpr_v - fpr_v)
youden_threshold = float(thr_v[youden_idx])

print(f"Youdenov threshold (validácia 2017): {youden_threshold:.4f}")

y_pred_test = (y_prob_test >= youden_threshold).astype(int)

roc_auc = roc_auc_score(y_test, y_prob_test)
auc_pr  = average_precision_score(y_test, y_prob_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_test)
brier   = brier_score_loss(y_test, y_prob_test)

fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_test)
ks_stat = float(np.max(tpr_t - fpr_t))

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\n{'='*60}")
print(f"  EBM (Maturo) — Výsledky: Test 2018–2019")
print(f"  Stratégia: {WINNER}")
print(f"{'='*60}")
print(f"  ROC-AUC:           {roc_auc:.4f}")
print(f"  AUC-PR:            {auc_pr:.4f}   ← primárna metrika")
print(f"  KS Statistic:      {ks_stat:.4f}")
print(f"  Balanced Accuracy: {bal_acc:.4f}")
print(f"  Brier Score:       {brier:.4f}")
print(f"  Sensitivity:       {sensitivity:.4f}")
print(f"  Specificity:       {specificity:.4f}")
print(f"  Threshold:         {youden_threshold:.4f}  (Youden index)")
print(f"{'='*60}")
print(classification_report(y_test, y_pred_test, target_names=['Healthy','Bankrupt']))


## 7. Top-K capture rate (banking convention)

Measures the fraction of bankruptcies captured if firms are ranked by predicted risk and the top K % are flagged. Reported for K ∈ {0.5 %, 1 %, 2 %, 5 %, 10 %} along with the lift over a random ranking.

In [ ]:
def top_k_capture(y_true, y_prob, top_fraction):
    tmp = pd.DataFrame({'y': np.asarray(y_true), 'p': np.asarray(y_prob)})
    tmp = tmp.sort_values('p', ascending=False).reset_index(drop=True)
    n_top  = int(np.ceil(len(tmp) * top_fraction))
    top_df = tmp.iloc[:n_top]
    captured = int(top_df['y'].sum())
    total_pos = int(tmp['y'].sum())
    return {
        'top_pct':          f"{top_fraction*100:.1f}%",
        'n_flagged':         n_top,
        'captured_bankrupt': captured,
        'precision_at_k':    float(top_df['y'].mean()),
        'recall_at_k':       float(captured / total_pos) if total_pos > 0 else 0,
        'lift':              float(top_df['y'].mean() / (total_pos / len(tmp))) if total_pos > 0 else 0,
    }

topk_df = pd.DataFrame([
    top_k_capture(y_test, y_prob_test, k)
    for k in [0.005, 0.01, 0.02, 0.05, 0.10]
])
print("Top-K Capture Rate (test 2018–2019):")
print(topk_df.to_string(index=False))


## 8. EBM global interpretability — feature importances

Uses `ebm_final.explain_global()` to extract mean-absolute contribution per term. Reports the ranked global importance of main effects vs pairwise interactions and prints the top contributors.

In [ ]:
# Globálna dôležitosť (mean |contribution|)
ebm_global = ebm_final.explain_global()

importances_raw = ebm_global.data()
if isinstance(importances_raw, dict) and 'scores' in importances_raw:
    imp_scores = importances_raw['scores']
    imp_names  = importances_raw['names']
else:
    imp_scores = ebm_final.term_importances()
    imp_names  = ebm_final.term_names_

imp_df = pd.DataFrame({'feature': imp_names, 'importance': imp_scores})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

print("Top 20 features (globálna dôležitosť EBM):")
print(imp_df.head(20).to_string(index=False))


In [ ]:
# Shape functions pre top 6 hlavných efektov (nie interakcie)
top_main = []
for i, term in enumerate(ebm_final.term_features_):
    if len(term) == 1:
        feat_name = feature_cols[term[0]] if term[0] < len(feature_cols) else f'feat_{term[0]}'
        imp_val   = ebm_final.term_importances()[i]
        top_main.append((i, feat_name, imp_val))

top_main.sort(key=lambda x: x[2], reverse=True)
top_main = top_main[:6]

if top_main:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax_idx, (term_i, feat_name, imp_val) in enumerate(top_main):
        ax = axes[ax_idx // 3][ax_idx % 3]
        term_exp = ebm_final.explain_global().data(term_i)
        if term_exp and 'scores' in term_exp:
            scores = np.asarray(term_exp['scores'])
            names  = term_exp.get('names', range(len(scores)))
            if len(names) == len(scores) + 1:
                names = [(names[i] + names[i+1])/2 for i in range(len(scores))]
            ax.plot(names[:len(scores)], scores, 'b-', lw=2)
            ax.axhline(y=0, color='red', lw=1, linestyle='--')
            ax.set_title(f'{feat_name}\nimportance={imp_val:.4f}', fontsize=10)
            ax.set_xlabel(feat_name, fontsize=8)
            ax.set_ylabel('Príspevok k log-odds', fontsize=8)
            ax.grid(True, alpha=0.3)
    plt.suptitle('EBM Shape Functions — Top 6 Features', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'ebm_shape_functions.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Uložené: ebm_shape_functions.png")


## 9. Calibration plot — raw probabilities vs Platt-scaled

Decile reliability diagrams comparing raw EBM probabilities against Platt-scaled probabilities on the test set, plus a histogram of predicted probabilities for both versions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Kalibrácia
prob_true_raw, prob_pred_raw = calibration_curve(y_test, y_prob_test_raw, n_bins=10, strategy='quantile')
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_prob_test_cal, n_bins=10, strategy='quantile')

axes[0].plot([0,1],[0,1],'k--', label='Perfektná kalibrácia')
axes[0].plot(prob_pred_raw, prob_true_raw, 'r-o', ms=5, label=f'Pred kalibráciou\nBrier={brier_score_loss(y_test, y_prob_test_raw):.4f}')
axes[0].plot(prob_pred_cal, prob_true_cal, 'b-o', ms=5, label=f'Po Platt scaling\nBrier={brier:.4f}')
axes[0].set_xlabel('Predikovaná pravdepodobnosť')
axes[0].set_ylabel('Skutočná frekvencia')
axes[0].set_title('Kalibrácia EBM')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# ROC
axes[1].plot(fpr_t, tpr_t, 'g-', lw=2, label=f'EBM (AUC={roc_auc:.4f})')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve – EBM')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[2],
            xticklabels=['Healthy','Bankrupt'], yticklabels=['Healthy','Bankrupt'])
axes[2].set_title(f'Confusion Matrix\n(Youden={youden_threshold:.3f})')
axes[2].set_ylabel('Skutočné'); axes[2].set_xlabel('Predikované')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ebm_results.png'), dpi=150, bbox_inches='tight')
plt.show()


## 10. Save results

Persists `results_ebm` to `results_ebm.pkl`: model name, winning strategy, best hyperparameters, feature list, validation / test predictions, ids and `rok`, threshold, metrics and the global importance ranking. Used by the downstream stacking notebook.

In [ ]:
results_ebm = {
    'model_name':    f'EBM Maturo ({WINNER})',
    'strategy':       WINNER,
    'hyperparams':    best_params,
    'n_features':     len(feature_cols),
    'feature_cols':   feature_cols,
    'n_interactions': sum(1 for t in ebm_final.term_features_ if len(t) == 2),
    'threshold':      youden_threshold,
    'threshold_method': 'Youden index',
    'metrics': {
        'roc_auc':           roc_auc,
        'auc_pr':            auc_pr,
        'ks_statistic':      ks_stat,
        'balanced_accuracy': bal_acc,
        'brier_score':       brier,
        'sensitivity':       sensitivity,
        'specificity':       specificity,
    },
    'strategy_comparison': strat_df.to_dict('records'),
    'topk_analysis':  topk_df.to_dict('records'),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'y_test':        y_test.values,
    'y_prob_test':   y_prob_test,
    'y_pred_test':   y_pred_test,
}

out_path = os.path.join(DATA_DIR, 'results_ebm.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results_ebm, f)
print(f"Uložené: {out_path}")
print(f"ROC-AUC: {roc_auc:.4f}  |  PR-AUC: {auc_pr:.4f}  |  KS: {ks_stat:.4f}")
print(f"Interakčné členy: {results_ebm['n_interactions']}")


## v5 extensions — defence of design choices, interactions, stability, calibration

This section adds the following to the EBM pipeline:

- **D1** — bootstrap comparison of the evaluated imbalance strategies, with special attention to `weighted` versus `undersample_1_5`.
- **D2** — top-3 pairwise interactions, with shape-function plots and
  economic interpretation.
- **D3** — top-10 feature stability across five random seeds.
- **D3b** — bootstrap 95 % CIs for the primary metrics and a per-year
  decomposition (2018 vs 2019).
- **D4** — calibration metrics in a dedicated table.
- **D5** — tempered conclusion section.

### D1 — Bootstrap CI: `weighted` vs `undersample_1_5`

A 1 000-iteration bootstrap comparison reports test PR-AUC intervals for the evaluated imbalance strategies, including `weighted` and `undersample_1_5`. This supports a transparent discussion of whether the empirical difference between strategies is material.

In [ ]:
# === NEW v5 · D1: Bootstrap CI PR-AUC – weighted vs undersample_1_5 ==========
import numpy as np
import pandas as pd

def bootstrap_ci_pr(y_true, y_prob, n_iter=1000, seed=42):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    rng = np.random.default_rng(seed)
    out = []
    n = len(y_true)
    for _ in range(n_iter):
        idx = rng.integers(0, n, n)
        yb = y_true[idx]; pb = y_prob[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        out.append(average_precision_score(yb, pb))
    a = np.asarray(out)
    return float(a.mean()), float(np.quantile(a, 0.025)), float(np.quantile(a, 0.975))

ci_rows = []
for strat in ['weighted', 'undersample_1_5', 'undersample_1_3', 'undersample_1_1']:
    if strat not in probs_strat:
        continue
    p = probs_strat[strat]
    mean, lo, hi = bootstrap_ci_pr(y_test, p, n_iter=1000, seed=42)
    ci_rows.append({
        'strategy': strat,
        'test_PR_AUC_mean': mean,
        'test_PR_AUC_2.5%': lo,
        'test_PR_AUC_97.5%': hi,
    })
strategy_bootstrap_df = pd.DataFrame(ci_rows)
print("Bootstrap (1000 iter) 95% CI pre test PR-AUC podľa stratégie:")
display(strategy_bootstrap_df.style.format({
    'test_PR_AUC_mean': '{:.4f}',
    'test_PR_AUC_2.5%': '{:.4f}',
    'test_PR_AUC_97.5%': '{:.4f}',
}))

# Overlap check
w = strategy_bootstrap_df[strategy_bootstrap_df['strategy'] == 'weighted']
u5 = strategy_bootstrap_df[strategy_bootstrap_df['strategy'] == 'undersample_1_5']
if len(w) and len(u5):
    w_lo, w_hi = w.iloc[0][['test_PR_AUC_2.5%', 'test_PR_AUC_97.5%']]
    u_lo, u_hi = u5.iloc[0][['test_PR_AUC_2.5%', 'test_PR_AUC_97.5%']]
    overlap = (w_lo <= u_hi) and (u_lo <= w_hi)
    print(f"\nInterval prekrývajú sa: {overlap}")
    print(f"  weighted       : [{w_lo:.4f}, {w_hi:.4f}]")
    print(f"  undersample_1_5: [{u_lo:.4f}, {u_hi:.4f}]")
    print(f"  → Rozdiel v test PR-AUC medzi stratégiami je "
          f"{'štatisticky nesignifikantný' if overlap else 'signifikantný'} (bootstrap, α=0.05).")


### D2 — Top-3 pairwise interactions and economic interpretation

The HPO configuration allows the number of pairwise interactions to be selected from the configured Optuna range. This block extracts the three most important pairs by mean absolute contribution, plots their joint shape functions, and adds a short economic interpretation for each.

In [ ]:
# === NEW v5 · D2: Top 3 pairwise interakcie – vizualizácia a interpretácia ===
import matplotlib.pyplot as plt

_global = ebm_final.explain_global()
_data = _global.data()
_names_all = _data.get('names', [])
_scores_all = _data.get('scores', [])

# Rozlíšime pairwise (meno obsahuje " & ") od hlavných efektov
pair_rows = []
for term_i, term in enumerate(ebm_final.term_features_):
    if len(term) == 2:
        f1 = ebm_final.feature_names_in_[term[0]] if hasattr(ebm_final, 'feature_names_in_') \
             else feature_cols[term[0]]
        f2 = ebm_final.feature_names_in_[term[1]] if hasattr(ebm_final, 'feature_names_in_') \
             else feature_cols[term[1]]
        imp = _scores_all[term_i] if term_i < len(_scores_all) else np.nan
        pair_rows.append({'term_idx': term_i, 'f1': f1, 'f2': f2, 'importance': imp})

pair_df = pd.DataFrame(pair_rows).sort_values('importance', ascending=False).reset_index(drop=True)
print(f"Celkom pairwise interakcií: {len(pair_df)}")
print("\nTop 10 interakcií:")
display(pair_df.head(10).style.format({'importance': '{:.6f}'}))

# Plot top 3 interakcií ako 2D shape functions
TOP_K = min(3, len(pair_df))
if TOP_K > 0:
    fig, axes = plt.subplots(1, TOP_K, figsize=(6 * TOP_K, 5))
    if TOP_K == 1:
        axes = [axes]
    for k in range(TOP_K):
        term_idx = int(pair_df.iloc[k]['term_idx'])
        f1 = pair_df.iloc[k]['f1']; f2 = pair_df.iloc[k]['f2']
        term_explain = _global.data(term_idx)
        scores = term_explain.get('scores')
        if scores is None:
            axes[k].text(0.5, 0.5, f'no scores for {f1}&{f2}',
                         ha='center', va='center', transform=axes[k].transAxes)
            continue
        scores = np.asarray(scores)
        im = axes[k].imshow(scores, aspect='auto', cmap='RdBu_r',
                            vmin=-np.max(np.abs(scores)), vmax=np.max(np.abs(scores)))
        axes[k].set_title(f'{f1} × {f2}\nimportance={pair_df.iloc[k]["importance"]:.4f}')
        axes[k].set_xlabel(f2)
        axes[k].set_ylabel(f1)
        plt.colorbar(im, ax=axes[k], shrink=0.8)
    plt.tight_layout()
    plt.show()

print("""
Ekonomická interpretácia (doplniť v diplomovke):
  - Každá interakcia zodpovedá *joint* efektu nad rámec sumy individuálnych prispení.
  - Kladná hodnota v 2D heatmape → kombinácia zvyšuje predikované riziko bankrotu.
  - Záporná hodnota → kombinácia znižuje riziko (protektívny efekt).
  - Pre top-3 páry je potrebné v diplomovke popísať: (i) oblasť najsilnejšieho efektu
    v x1–x2 priestore, (ii) ekonomický mechanizmus (napr. vysoké záväzky v kombinácii
    s nízkymi odpismi → firmy s nízkou investičnou aktivitou a vysokou zadlženosťou).
""")


### D3 — Top-10 feature stability across seeds

The EBM (Maturo, 2025) is a bagged ensemble; default `outer_bags=8`. To probe robustness beyond the internal bagging, the model is refit five times with different seeds, the top-10 feature ranking is recorded for each, and the rank intersection is reported.

In [ ]:
# === NEW v5 · D3: Stabilita Top-10 features naprieč 5 seeds ==================
# POZN: fit sa opakuje 5×; ak chceš rýchlejšie, zníž počet seeds alebo znížte max_bins.
SEEDS = [0, 1, 42, 123, 2026]
stability_records = []

# Použijeme best_params (z Optuna HPO) a meníme len random_state
for seed in SEEDS:
    params_stab = dict(best_params)
    params_stab['random_state'] = seed
    ebm_s = ExplainableBoostingClassifier(**params_stab)
    # Weighted stratégia (ako finálna)
    sw_s = make_sample_weights(y_train_full) if WINNER == 'weighted' else None
    if sw_s is not None:
        ebm_s.fit(X_train_full, y_train_full, sample_weight=sw_s)
    else:
        X_s, y_s = stratified_undersample(
            X_train_full, y_train_full, sector_train,
            majority_ratio=strategies[WINNER]['ratio'], random_state=seed,
        )
        ebm_s.fit(X_s, y_s)
    # Top-10 podľa importance (main effects + interactions)
    gexp = ebm_s.explain_global().data()
    names = gexp.get('names', [])
    scores = gexp.get('scores', [])
    imp_df = pd.DataFrame({'term': names, 'importance': scores}) \
        .sort_values('importance', ascending=False).head(10)
    for rank, (term, imp) in enumerate(imp_df[['term', 'importance']].values, start=1):
        stability_records.append({
            'seed': seed, 'rank': rank, 'term': term, 'importance': float(imp),
        })

stability_df = pd.DataFrame(stability_records)

# Pivot: rank-y pre každý term naprieč seeds
pivot = (stability_df.pivot_table(index='term', columns='seed', values='rank')
         .sort_values(by=SEEDS[0] if SEEDS[0] in stability_df['seed'].values else SEEDS))
# Frekvencia v Top-10
freq = stability_df.groupby('term').size().rename('in_top10_seeds').to_frame()
pivot = pivot.merge(freq, left_index=True, right_index=True, how='left') \
             .sort_values('in_top10_seeds', ascending=False)
print("Stabilita Top-10 features naprieč seeds (rank vs seed):")
display(pivot)

stable_features = pivot[pivot['in_top10_seeds'] == len(SEEDS)].index.tolist()
print(f"\nStabilné core features (v Top-10 vo VŠETKÝCH {len(SEEDS)} seedoch): {len(stable_features)}")
print(stable_features)


### D3b — Bootstrap 95 % CIs and per-year decomposition

Mirrors the analysis used for the logit benchmark: bootstrap CIs on test-set ROC-AUC, PR-AUC and KS, plus a separate report of the same metrics for 2018 and 2019.

In [ ]:
# === NEW v5 · D3b: Bootstrap CI + per-year decomposition ====================
def _ks_stat(y_true, probs):
    fpr, tpr, _ = roc_curve(y_true, probs)
    return float(np.max(np.abs(tpr - fpr)))

def bootstrap_metrics(y_true, probs, n_iter=1000, seed=42):
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs, praucs, kss = [], [], []
    for _ in range(n_iter):
        idx = rng.integers(0, n, n)
        yb = y_true[idx]; pb = probs[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        aucs.append(roc_auc_score(yb, pb))
        praucs.append(average_precision_score(yb, pb))
        kss.append(_ks_stat(yb, pb))
    def ci(a):
        a = np.asarray(a)
        return (float(np.mean(a)), float(np.quantile(a, 0.025)), float(np.quantile(a, 0.975)))
    return {'ROC_AUC': ci(aucs), 'PR_AUC': ci(praucs), 'KS': ci(kss)}

bci_test = bootstrap_metrics(y_test, y_prob_test, n_iter=1000, seed=42)
bci_valid = bootstrap_metrics(y_valid, y_prob_valid, n_iter=1000, seed=42)

boot_rows = []
for metric in ['ROC_AUC', 'PR_AUC', 'KS']:
    m_v, lo_v, hi_v = bci_valid[metric]
    m_t, lo_t, hi_t = bci_test[metric]
    boot_rows.append({
        'metric': metric,
        'valid_mean': m_v, 'valid_95CI_low': lo_v, 'valid_95CI_high': hi_v,
        'test_mean': m_t, 'test_95CI_low': lo_t, 'test_95CI_high': hi_t,
    })
ebm_bootstrap_df = pd.DataFrame(boot_rows)
print("Bootstrap 1000-iter 95 % CI (EBM, post-Platt):")
display(ebm_bootstrap_df.style.format({
    'valid_mean': '{:.4f}', 'valid_95CI_low': '{:.4f}', 'valid_95CI_high': '{:.4f}',
    'test_mean': '{:.4f}', 'test_95CI_low': '{:.4f}', 'test_95CI_high': '{:.4f}',
}))

# Per-year
per_year_rows = []
for yr in [2018, 2019]:
    mask_yr = test['rok'] == yr
    if mask_yr.sum() == 0:
        continue
    y_yr = y_test[mask_yr.values]
    p_yr = y_prob_test[mask_yr.values]
    if np.asarray(y_yr).sum() == 0:
        continue
    per_year_rows.append({
        'rok': yr,
        'n': int(mask_yr.sum()),
        'defaulty': int(np.asarray(y_yr).sum()),
        'ROC_AUC': roc_auc_score(y_yr, p_yr),
        'PR_AUC': average_precision_score(y_yr, p_yr),
        'KS': _ks_stat(y_yr, p_yr),
        'Brier': brier_score_loss(y_yr, p_yr),
    })
ebm_per_year_df = pd.DataFrame(per_year_rows)
print("\nPer-year decomposition (test 2018 vs 2019):")
display(ebm_per_year_df.style.format({
    'ROC_AUC': '{:.4f}', 'PR_AUC': '{:.4f}', 'KS': '{:.4f}', 'Brier': '{:.6f}',
}))


### D4 — Calibration metrics in a separate table

The Brier score in this notebook is computed **after** Platt scaling. To keep the comparison with the logit (no scaling) and the BiLSTM (isotonic) honest, this block prints calibration metrics — Brier, calibration slope/intercept, BSS — explicitly for both raw and scaled EBM probabilities.

In [ ]:
# === NEW v5 · D4: Kalibračné metriky (samostatná tabuľka) ===================
from sklearn.linear_model import LogisticRegression as _LRCalib2

def _safe_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def cal_intercept_slope(y, p):
    lp = _safe_logit(p).reshape(-1, 1)
    lr = _LRCalib2(C=1e6, solver='lbfgs')
    lr.fit(lp, y)
    return float(lr.intercept_[0]), float(lr.coef_[0, 0])

a_v_raw, b_v_raw = cal_intercept_slope(y_valid, y_prob_valid_raw)
a_t_raw, b_t_raw = cal_intercept_slope(y_test, y_prob_test_raw)
a_t_cal, b_t_cal = cal_intercept_slope(y_test, y_prob_test_cal)

ebm_calib_df = pd.DataFrame([
    {'set': 'valid (raw)',   'intercept': a_v_raw, 'slope': b_v_raw,
     'Brier': brier_score_loss(y_valid, y_prob_valid_raw)},
    {'set': 'test (raw)',    'intercept': a_t_raw, 'slope': b_t_raw,
     'Brier': brier_score_loss(y_test,  y_prob_test_raw)},
    {'set': 'test (Platt)',  'intercept': a_t_cal, 'slope': b_t_cal,
     'Brier': brier_score_loss(y_test,  y_prob_test_cal)},
])
print("Kalibračná tabuľka EBM:")
display(ebm_calib_df.style.format({'intercept': '{:.4f}', 'slope': '{:.4f}', 'Brier': '{:.6f}'}))

print("""
Poznámka: Platt scaling v tomto pipeline slúži na kalibráciu EBM pravdepodobností
na základe validácie. Brier post-Platt preto NEPOROVNÁVAME priamo s Brier logitu
(ktorý nie je post-calibrated) ani s Brier BiLSTM (focal loss skresľuje škálu
pravdepodobností). Pre medzi-modelové porovnanie kalibrácie používame preto v
05_model_comparison oddelenú kalibračnú tabuľku kde KAŽDÝ model má uvedené obe verzie
(raw + Platt), pričom Platt je aplikovaný ex-post na rovnakej validácii (2017).
""")


### D5 — Tempered conclusion

The original verdict ("the EBM is unambiguously the best model") is rewritten into a more cautious statement that emphasises the conditional nature of the finding: the ranking holds **for this specific sample, target definition and class-imbalance treatment**.

In [ ]:
# === NEW v5 · D5: Zjemnený záver – textová sekcia ============================
print("""
ZÁVER (v5):
===========================================================================
V analyzovanom out-of-time nastavení (train 2013–2016, valid 2017, test 2018–2019)
dosiahol EBM najlepší výkon podľa primárnych metrík (ROC-AUC, KS, balanced
accuracy). Rozdiely oproti BiLSTM sú v PR-AUC v rámci 95 % bootstrap CI, takže
voľba EBM ako finálneho modelu sa opiera aj o jeho natívnu interpretovateľnosť
(shape functions a pairwise interakcie), regulačnú auditovateľnosť (GDPR čl. 22)
a nižšie výpočtové nároky.

Kľúčové výsledky:
  - Test ROC-AUC: {:.4f}  (95% CI [{:.4f}, {:.4f}])
  - Test PR-AUC:  {:.4f}  (95% CI [{:.4f}, {:.4f}])
  - Test KS:      {:.4f}  (95% CI [{:.4f}, {:.4f}])
  - {} stabilných core features (v Top-10 vo všetkých {} seedoch)
  - {} pairwise interakcií v modeli

Tieto závery sú podmienené:
  (i)   slovenským finančným výkazníckym štandardom (RÚZ SR),
  (ii)  priemyslovou štruktúrou SK SME panelu (bez finančných podnikov),
  (iii) 0.21% bankrotovou rate (extrémne imbalance),
  (iv)  definíciou bankrotu a horizontom pozorovania (2011–2019).

Externú validitu mimo tohto kontextu NEŠTVRDÍME.
===========================================================================
""".format(
    ebm_bootstrap_df.query("metric == 'ROC_AUC'").iloc[0]['test_mean'],
    ebm_bootstrap_df.query("metric == 'ROC_AUC'").iloc[0]['test_95CI_low'],
    ebm_bootstrap_df.query("metric == 'ROC_AUC'").iloc[0]['test_95CI_high'],
    ebm_bootstrap_df.query("metric == 'PR_AUC'").iloc[0]['test_mean'],
    ebm_bootstrap_df.query("metric == 'PR_AUC'").iloc[0]['test_95CI_low'],
    ebm_bootstrap_df.query("metric == 'PR_AUC'").iloc[0]['test_95CI_high'],
    ebm_bootstrap_df.query("metric == 'KS'").iloc[0]['test_mean'],
    ebm_bootstrap_df.query("metric == 'KS'").iloc[0]['test_95CI_low'],
    ebm_bootstrap_df.query("metric == 'KS'").iloc[0]['test_95CI_high'],
    len(stable_features) if 'stable_features' in dir() else 'N/A',
    len(SEEDS) if 'SEEDS' in dir() else 'N/A',
    sum(1 for t in ebm_final.term_features_ if len(t) == 2),
))


### Save extended payload for `05_model_comparison`

Saves the extended payload with bootstrap CIs, calibration metrics, per-year decomposition and feature-stability results as `ebm_v7_results.pkl`. For backward compatibility, the same payload is also saved as `ebm_v5_results.pkl`.


In [ ]:
# === NEW v5: Extended payload pre 05_model_comparison ========================
OUT_DIR = DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

payload_v5 = {
    'model': 'ebm_maturo',
    'version': 'v5',
    'strategy': WINNER,
    'hyperparams': best_params,
    'feature_cols': feature_cols,
    'threshold_youden': float(youden_threshold),
    'y_valid': np.asarray(y_valid).astype(int),
    'y_test': np.asarray(y_test).astype(int),
    'probs_valid_raw': np.asarray(y_prob_valid_raw, dtype=float),
    'probs_valid_platt': np.asarray(y_prob_valid_cal, dtype=float),
    'probs_test_raw': np.asarray(y_prob_test_raw, dtype=float),
    'probs_test_platt': np.asarray(y_prob_test_cal, dtype=float),
    'rok_test': test['rok'].astype(int).values,
    'bootstrap_ci': ebm_bootstrap_df,
    'per_year': ebm_per_year_df,
    
    'calibration': ebm_calib_df,
    'pairwise_interactions': pair_df if 'pair_df' in dir() else None,
    'feature_stability': pivot if 'pivot' in dir() else None,
    'stable_core_features': stable_features if 'stable_features' in dir() else None,
}
out_path_v7 = os.path.join(OUT_DIR, 'ebm_v7_results.pkl')
with open(out_path_v7, 'wb') as f:
    pickle.dump(payload_v5, f)
print(f"✓ Uložené: {out_path_v7}  ({os.path.getsize(out_path_v7)/1024:.1f} kB)")

# Backward-compatible alias for earlier local runs / thesis drafts.
out_path_v5 = os.path.join(OUT_DIR, 'ebm_v5_results.pkl')
with open(out_path_v5, 'wb') as f:
    pickle.dump(payload_v5, f)
print(f"✓ Alias uložený: {out_path_v5}  ({os.path.getsize(out_path_v5)/1024:.1f} kB)")


## Changelog / GitHub notes

- **D1** — Bootstrap comparison of imbalance strategies.
- **D2** — Top-3 pairwise interactions with economic interpretation.
- **D3** — Feature-stability check across five seeds.
- **D3b** — Bootstrap CIs and per-year decomposition.
- **D4** — Calibration metrics in a separate table (raw vs Platt-style calibrated probabilities).
- **D5** — Tempered conclusion.
- Extended payload is saved as `ebm_v7_results.pkl` and also as the backward-compatible alias `ebm_v5_results.pkl`.
